# 04 — Evaluate the base and fine-tuned models

This notebook creates a small, reproducible comparison harness. Dataset loading and metric helpers are laptop-safe. Model calls are disabled: point the client at two OpenAI-compatible endpoints, or adapt the local Hugging Face example after exporting the NeMo adapter to a compatible format.

Do not use the test split to choose training hyperparameters. The sample set demonstrates the workflow only; three examples cannot establish model quality.

In [ ]:
from pathlib import Path
import json
import os
import re
from statistics import mean

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_jsonl(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8-sig') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def prompt_and_reference(record: dict) -> tuple[str, str]:
    if 'input' in record:
        return record['input'], record['output']
    assistant_indices = [i for i, message in enumerate(record['messages']) if message['role'] == 'assistant']
    answer_index = assistant_indices[-1]
    prompt = '\n'.join(
        f"{message['role'].capitalize()}: {message['content']}"
        for message in record['messages'][:answer_index]
    )
    return prompt, record['messages'][answer_index]['content']

test_records = load_jsonl(PROJECT_ROOT / 'data' / 'test.jsonl')
eval_cases = [dict(zip(('prompt', 'reference'), prompt_and_reference(record))) for record in test_records]
print(json.dumps(eval_cases, indent=2, ensure_ascii=False))

## Qualitative comparison

Use deterministic decoding first (`temperature=0`) and compare the same prompts. For each pair, record:

- correctness and factual support;
- instruction following and requested length/format;
- relevance and unnecessary content;
- safety or privacy problems;
- whether the adapter improved domain behavior without obvious regressions.

Blind the model labels when possible. Preserve raw outputs and inference settings so another person can audit the judgment.

In [ ]:
CALL_ENDPOINTS = False
BASE_URL = 'http://127.0.0.1:30000/v1'
TUNED_URL = 'http://127.0.0.1:30001/v1'
BASE_MODEL_NAME = 'Qwen/Qwen3-1.7B'
TUNED_MODEL_NAME = 'qwen3-1.7b-nemo-lora'

def query_openai_compatible(base_url: str, model: str, prompt: str) -> str:
    from openai import OpenAI
    client = OpenAI(base_url=base_url, api_key=os.getenv('MODEL_API_KEY', 'not-required'))
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=160,
    )
    return response.choices[0].message.content or ''

results = []
if CALL_ENDPOINTS:
    for case in eval_cases:
        results.append({
            **case,
            'base_response': query_openai_compatible(BASE_URL, BASE_MODEL_NAME, case['prompt']),
            'tuned_response': query_openai_compatible(TUNED_URL, TUNED_MODEL_NAME, case['prompt']),
        })
else:
    results = [{**case, 'base_response': None, 'tuned_response': None} for case in eval_cases]
    print('Endpoint calls skipped. Configure both URLs and set CALL_ENDPOINTS=True.')

print(json.dumps(results, indent=2, ensure_ascii=False))

## Optional local base-model path

For a small local baseline, reuse the Qwen2.5 generation cell from Notebook 1. A NeMo LoRA checkpoint is not automatically a Hugging Face PEFT directory. Export or merge it using the conversion workflow supported by the exact NeMo release before attempting to load it with `transformers`/`peft`. Keep model IDs and formats in the result metadata; otherwise the comparison is not reproducible.

## Automated-evaluation placeholders

Exact match is useful only for constrained answers. Token overlap is a smoke metric, not a semantic quality measure. Replace or supplement these with task-specific tests, structured-output validation, safety checks, and a vetted evaluation framework. An LLM-as-judge can help at scale but needs rubric validation, bias checks, version pinning, and human spot review.

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(re.findall(r'[a-z0-9]+', text.lower()))

def exact_match(prediction: str, reference: str) -> float:
    return float(normalize_text(prediction) == normalize_text(reference))

def token_f1(prediction: str, reference: str) -> float:
    prediction_tokens = normalize_text(prediction).split()
    reference_tokens = normalize_text(reference).split()
    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)
    common = 0
    remaining = reference_tokens.copy()
    for token in prediction_tokens:
        if token in remaining:
            common += 1
            remaining.remove(token)
    precision = common / len(prediction_tokens)
    recall = common / len(reference_tokens)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

completed_results = [row for row in results if row['base_response'] is not None and row['tuned_response'] is not None]
if completed_results:
    for label in ['base_response', 'tuned_response']:
        print(label, {
            'exact_match': mean(exact_match(row[label], row['reference']) for row in completed_results),
            'token_f1': mean(token_f1(row[label], row['reference']) for row in completed_results),
        })
else:
    print('No generated responses yet; metric functions are ready for endpoint results.')

In [ ]:
SAVE_RESULTS = False
RESULTS_PATH = PROJECT_ROOT / 'outputs' / 'evaluation' / 'base_vs_tuned.jsonl'

if SAVE_RESULTS:
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with RESULTS_PATH.open('w', encoding='utf-8', newline='\n') as handle:
        for row in results:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    print('Saved:', RESULTS_PATH)
else:
    print('Results were not written. Set SAVE_RESULTS=True after generation.')

## Next

Notebook 5 serves a compatible model with SGLang and uses the same OpenAI-style client interface.